In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm


In [23]:
# =========================
# LOAD ALL PARQUETS
# =========================

parquet_files = glob.glob(
    "/Users/lena/PyCharmMiscProject/outputs_test/poses/*.parquet"
)

dfs = []

for path in parquet_files:

    df = pd.read_parquet(path)

    video_key = os.path.basename(path).replace(".parquet", "")

    df["video_key"] = video_key

    dfs.append(df)

poses = pd.concat(dfs, ignore_index=True)

print("POSES:", poses.shape)


POSES: (132187, 59)


In [24]:


# =========================
# SORT
# =========================

poses = poses.sort_values(
    ["video_key", "fighter", "frame"]
).reset_index(drop=True)

# =========================
# SPEED FEATURES
# =========================

def add_speed_feature(df, kp_idx, name):

    x_col = f"kp_{kp_idx}_x"
    y_col = f"kp_{kp_idx}_y"

    dx = df.groupby(
        ["video_key", "fighter"]
    )[x_col].diff()

    dy = df.groupby(
        ["video_key", "fighter"]
    )[y_col].diff()

    speed = np.sqrt(dx**2 + dy**2)

    df[f"{name}_speed"] = speed

    accel = speed.groupby(
        [df["video_key"], df["fighter"]]
    ).diff()

    df[f"{name}_accel"] = accel

    return df

# wrists
poses = add_speed_feature(poses, 9, "left_wrist")
poses = add_speed_feature(poses, 10, "right_wrist")

# elbows
poses = add_speed_feature(poses, 7, "left_elbow")
poses = add_speed_feature(poses, 8, "right_elbow")



In [25]:
# =========================
# RESTORE BBOX FROM KEYPOINTS
# =========================

x_cols = [c for c in poses.columns if c.endswith("_x")]
y_cols = [c for c in poses.columns if c.endswith("_y")]

poses["bbox_x1"] = poses[x_cols].min(axis=1)
poses["bbox_x2"] = poses[x_cols].max(axis=1)

poses["bbox_y1"] = poses[y_cols].min(axis=1)
poses["bbox_y2"] = poses[y_cols].max(axis=1)

# =========================
# CENTER FEATURES
# =========================

poses["center_x"] = (
    poses["bbox_x1"] + poses["bbox_x2"]
) / 2

poses["center_y"] = (
    poses["bbox_y1"] + poses["bbox_y2"]
) / 2

poses["bbox_w"] = (
    poses["bbox_x2"] - poses["bbox_x1"]
)

poses["bbox_h"] = (
    poses["bbox_y2"] - poses["bbox_y1"]
)


In [26]:
# OPPONENT FEATURES
# =========================

opp = poses.copy()

opp["fighter"] = opp["fighter"].map({
    "red": "blue",
    "blue": "red"
})

opp = opp.rename(columns={

    "center_x": "opp_center_x",
    "center_y": "opp_center_y",

    "kp_0_x": "opp_head_x",
    "kp_0_y": "opp_head_y",

    "left_wrist_speed": "opp_left_wrist_speed",
    "right_wrist_speed": "opp_right_wrist_speed"
})

merge_cols = [
    "video_key",
    "frame",
    "fighter",

    "opp_center_x",
    "opp_center_y",

    "opp_head_x",
    "opp_head_y",

    "opp_left_wrist_speed",
    "opp_right_wrist_speed"
]

poses = poses.merge(
    opp[merge_cols],
    on=["video_key", "frame", "fighter"],
    how="left"
)

In [27]:
# =========================
# DISTANCE FEATURES
# =========================

def euclidean(x1, y1, x2, y2):

    return np.sqrt(
        (x1 - x2)**2 +
        (y1 - y2)**2
    )

# left wrist -> opponent head
poses["left_wrist_to_head"] = euclidean(
    poses["kp_9_x"],
    poses["kp_9_y"],
    poses["opp_head_x"],
    poses["opp_head_y"]
)

# right wrist -> opponent head
poses["right_wrist_to_head"] = euclidean(
    poses["kp_10_x"],
    poses["kp_10_y"],
    poses["opp_head_x"],
    poses["opp_head_y"]
)

# center distance
poses["center_distance"] = euclidean(
    poses["center_x"],
    poses["center_y"],
    poses["opp_center_x"],
    poses["opp_center_y"]
)


In [28]:
# =========================
# ROLLING FEATURES
# =========================

grouped = poses.groupby(
    ["video_key", "fighter"]
)

poses["left_wrist_speed_mean_5"] = grouped[
    "left_wrist_speed"
].transform(
    lambda x: x.rolling(5).mean()
)

poses["right_wrist_speed_mean_5"] = grouped[
    "right_wrist_speed"
].transform(
    lambda x: x.rolling(5).mean()
)

poses["left_wrist_speed_max_5"] = grouped[
    "left_wrist_speed"
].transform(
    lambda x: x.rolling(5).max()
)

poses["right_wrist_speed_max_5"] = grouped[
    "right_wrist_speed"
].transform(
    lambda x: x.rolling(5).max()
)

In [29]:
# =========================
# CLEAN NaN
# =========================

poses = poses.fillna(0)

In [30]:
# =========================
# SAVE
# =========================

os.makedirs("data", exist_ok=True)

save_path = "data/final_test_dataset.parquet"

# FIX TYPES
string_cols = [
    "video_key",
    "fighter",
    "hand",
    "target",
    "effectiveness",
    "clear"
]

for col in string_cols:
    if col in poses.columns:
        poses[col] = poses[col].astype(str)

# =========================
# FIX OBJECT COLUMNS
# =========================

object_cols = poses.select_dtypes(include="object").columns

for col in object_cols:
    poses[col] = poses[col].astype(str)

poses.to_parquet(save_path)

print("\nDONE")
print("FINAL SHAPE:", poses.shape)
print("Saved:", save_path)


DONE
FINAL SHAPE: (132187, 88)
Saved: data/final_test_dataset.parquet
